# TRELLIS.2 on Google Colab with Gradio

Notebook for launching the [microsoft/TRELLIS.2](https://github.com/microsoft/TRELLIS.2) Gradio web demo on Colab.

Practical requirements: an NVIDIA GPU runtime with at least 24 GB of VRAM. The official README reports testing on A100/H100; on Colab, use A100/L4 when available and start with resolution 512.

## 1. Check GPU and environment

In Colab, select `Runtime > Change runtime type > GPU` before running the cells.

In [ ]:
!nvidia-smi
!python --version
!nvcc --version || true

## 2. Clone TRELLIS.2

In [ ]:
%cd /content
!rm -rf TRELLIS.2
!git clone -b main --recursive https://github.com/microsoft/TRELLIS.2.git
%cd /content/TRELLIS.2
!git submodule update --init --recursive

## 3. Install PyTorch and dependencies

This cell uses the repository's official `setup.sh` script. Compiling CUDA extensions may take several minutes.

Note: `setup.sh` may not stop if `flash-attn` fails on Colab/Python 3.12. The next cell verifies the attention backend and configures a fallback.

In [ ]:
%cd /content/TRELLIS.2
import os, subprocess, sys

os.environ["CUDA_HOME"] = os.environ.get("CUDA_HOME", "/usr/local/cuda")
os.environ["PATH"] = os.environ["CUDA_HOME"] + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

!pip install -U pip setuptools wheel
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!bash -lc '. ./setup.sh --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm'
print("Base installation completed. The attention backend is checked in the next cell.")

## 3b. Verify attention backend

TRELLIS.2 uses `flash_attn` by default. If it is not available, this cell installs `xformers` and forces `ATTN_BACKEND=xformers` and `SPARSE_ATTN_BACKEND=xformers` before launching the demo.

In [ ]:
import importlib.util, os, subprocess, sys

def module_exists(name):
    return importlib.util.find_spec(name) is not None

if module_exists("flash_attn"):
    os.environ["ATTN_BACKEND"] = "flash_attn"
    os.environ["SPARSE_ATTN_BACKEND"] = "flash_attn"
    print("flash_attn is available: using the flash_attn backend.")
else:
    print("flash_attn was not found: installing xformers and using the xformers fallback.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "xformers==0.0.29.post3", "--index-url", "https://download.pytorch.org/whl/cu124"])
    if not module_exists("xformers"):
        raise RuntimeError("xformers installation failed. Restart the Colab runtime and rerun the cells from the beginning.")
    os.environ["ATTN_BACKEND"] = "xformers"
    os.environ["SPARSE_ATTN_BACKEND"] = "xformers"
    print("xformers is available: backend configured.")

print("ATTN_BACKEND=", os.environ["ATTN_BACKEND"])
print("SPARSE_ATTN_BACKEND=", os.environ["SPARSE_ATTN_BACKEND"])

## 4. Hugging Face login required for DINOv3

TRELLIS.2 also uses `facebook/dinov3-vitl16-pretrain-lvd1689m`, which is a gated model. Before launching the demo, you need to:

1. Open https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m and request/accept access with your Hugging Face account.
2. Create a Hugging Face token with `Read` permission.
3. In Colab, add a secret named `HF_TOKEN` and enable it for this notebook.

Without approved access to that repo, Hugging Face returns `403 Forbidden` and the demo cannot start.

In [ ]:
import os
from huggingface_hub import hf_hub_download, login, whoami
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if not token:
    raise RuntimeError("Add a Colab secret named HF_TOKEN with a Hugging Face Read token.")

os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
login(token=token, add_to_git_credential=False)
print("Hugging Face login completed as:", whoami(token=token).get("name", "HF account"))

try:
    hf_hub_download(
        repo_id="facebook/dinov3-vitl16-pretrain-lvd1689m",
        filename="config.json",
        token=token,
    )
    print("DINOv3 access verified.")
except GatedRepoError as exc:
    raise RuntimeError(
        "Your account/token does not yet have access to facebook/dinov3-vitl16-pretrain-lvd1689m. "
        "Open https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m, request/accept access, "
        "then rerun this cell."
    ) from exc
except HfHubHTTPError as exc:
    raise RuntimeError(f"Hugging Face access check failed: {exc}") from exc

## 5. Adapt the demo for Colab

This cell sets `share=True` to create a public Gradio link and changes the default resolution to 512, which is more suitable for Colab runtimes.

In [ ]:
from pathlib import Path

repo_dir = Path("/content/TRELLIS.2")
if not repo_dir.exists():
    raise FileNotFoundError("Could not find /content/TRELLIS.2. Run the repository clone cell first.")

for filename in ["app.py", "app_texturing.py"]:
    path = repo_dir / filename
    if not path.exists():
        print(f"Skipping {filename}: file not found in {repo_dir}")
        continue
    text = path.read_text()
    text = text.replace('value="1024")', 'value="512")')
    text = text.replace('demo.launch(css=css, head=head)', 'demo.launch(css=css, head=head, share=True, debug=True)')
    text = text.replace('demo.launch()', 'demo.launch(share=True, debug=True)')
    path.write_text(text)

print("Gradio demo configured for Colab.")

## 6. Launch Image-to-3D

Run this cell and open the public `gradio.live` link printed in the output. On first launch, the app downloads the `microsoft/TRELLIS.2-4B` weights from Hugging Face.

In [ ]:
%cd /content/TRELLIS.2
import os
os.environ.setdefault("ATTN_BACKEND", "xformers")
os.environ.setdefault("SPARSE_ATTN_BACKEND", os.environ["ATTN_BACKEND"])
print("ATTN_BACKEND=", os.environ["ATTN_BACKEND"])
print("SPARSE_ATTN_BACKEND=", os.environ["SPARSE_ATTN_BACKEND"])
!ATTN_BACKEND=$ATTN_BACKEND SPARSE_ATTN_BACKEND=$SPARSE_ATTN_BACKEND python app.py

## Optional: PBR texturing demo

Stop the previous cell and run this one only if you want to upload an existing mesh and generate PBR textures from an image.

In [ ]:
%cd /content/TRELLIS.2
import os
os.environ.setdefault("ATTN_BACKEND", "xformers")
os.environ.setdefault("SPARSE_ATTN_BACKEND", os.environ["ATTN_BACKEND"])
!ATTN_BACKEND=$ATTN_BACKEND SPARSE_ATTN_BACKEND=$SPARSE_ATTN_BACKEND python app_texturing.py

## Quick notes

- If you see `No module named 'flash_attn'`, rerun the `3b. Verify attention backend` cell and restart the demo. You should see `ATTN_BACKEND=xformers`.
- If you see `403 Forbidden` for `facebook/dinov3-vitl16-pretrain-lvd1689m`, the token is missing or the account is not yet authorized for the gated DINOv3 model.
- If you hit a CUDA out-of-memory error, restart the runtime and use resolution 512, texture size 1024/2048, and a runtime with more VRAM.
- If `flash-attn` compilation fails, restart the runtime, rerun from cell 1, and verify that `nvcc` is available.
- The exported GLB can be downloaded directly from the Gradio interface.